# Introduction to Audio Machine Learning
## Week 5 Worksheet - Exceptions, Audio Data and Feature Extraction
-----
Welcome! In this notebook, we’ll introduce Python Exceptions, work with audio data, and extracting features from that audio.

There are weekly computer lab sessions, 12:10-13:00 , or 13:10-14:00, on Wednesdays in Computer Lab B.51, Hugh Robson Building. To get the most out of these sessions, I **strongly** recommend you work through the worksheet in your own time before attending the session.

## 1 - Exceptions in Python

In this section, you'll learn about Python Exceptions, which are a method for handling errors in Python. This will include
- `try`/`except` statements
- Catching specific exceptions
- Raising exceptions and using assert
- Audio file Input/Output (I/O) patterns for loading audio files

When working with files, there will often be problems, such as missing files or corrupted data). Exceptions allow you to anticipate these problems and tell your code what to do if these problems arise.

Exceptions are different to syntax errors. A syntax error occurs when an incorrect Python statement is detected, for example:

In [ ]:
for n in (1, 2):
    print(n)) # The extra bracket means this code will not run!

In [ ]:
def function():
a = unindented block # An indentation error is another kind of syntax error.

Here is another code that will produce an error:

In [ ]:
x = 0
y = 1
print(y/x)

This type of error is called an **exception error**. This means that the syntax of the Python code was correct, but an error was produced when running the code. The error message tells you what kind of error occurred, in this case the error is a `ZeroDivisionError` - Python doesn't know what to do when dividing by zero, so it **raises an exception error**.

There are many types of built-in Python exception errors, you can read about them [here](https://docs.python.org/3/library/exceptions.html).

### 1.1 `try` / `except` statements
If you anticipate that you might encounter a certain kind of exception error in your code, you can use a try/except statement
- `try`: run code that might raise an exception
- `except`: what to do if a specific exception occurs
Here is an example:

In [ ]:
y = 5
x = [5, 2, 7, 2, 0, 8]
y_over_x = [] # I want to make a list of y/x for every value of x
for item in x:
    try:
        y_over_x.append(y/item) # Try and find y/x
    except ZeroDivisionError: # This statement runs if a ZeroDivisionError exception is raised 
        y_over_x.append('infinity') 
        
print(y_over_x)

The above code now handles the ZeroDivisionError by appending 'infinity' to the list instead. If the exception is not handled like this, the program will crash.

---
### ✏️✏️ Exercise 1: Catching the Error✏️✏️
---

The following code attempts to convert a list of strings into a list of integers, using the `int()` function. Currently the code will crash if it encounters a string that the `int()` function can't handle.

Use a `try` / `except` statement to catch this kind of error. Look at the error message to see what type of Error is raised. Write an except statement that appends the value `None` to the list if that type of error is raised.


In [ ]:
values = ["42", "3.14", "abc", "-7"]
result = []
# Your code here
for s in values:
    result.append(int(s))

In [ ]:
# Check your answer
assert result == [42, None, None, -7], f"Got {result}"
print("Correct!")

-----

### 1.2 Catching specific Exceptions
You should always try to catch narrowest exception you expect. Common exceptions you might encounter are:
- ValueError: wrong value (e.g., "abc" to int)
- TypeError: wrong type (e.g., adding int to str)
- FileNotFoundError: missing file
- ZeroDivisionError: division by zero

It is possible not to specify the error type, but this poor practice and can hide bugs in your code:

In [ ]:
# Bad: catches everything (can hide real bugs)
x = 'abc'
try:
    y = int(x)
except Exception:
    print('Some error occurred!')

In [ ]:
# Better: be specific about error types
try:
    y = int(x)
except TypeError:
    print('TypeError occurred!')
except ValueError:
    print('ValueError occurred!')

---
### ✏️✏️ Exercise 2: Catching Specific Exceptions ✏️✏️
---

There are two different types of errors that will occur in the below for-loop. Write a try/except block which catches the two error types, and appends a string describing the error to the result list.



In [ ]:
values = ["42", "3.14", "abc", "-7", None]
result = []
# Your code here
for s in values:
    result.append(int(s))

---
### 1.3 `else` and `finally`

- Use `else` when you want to run code only if the try block succeeded (no Exceptions raised).
- Use `finally` for cleanup, runs regardless of success or failure. 

In [ ]:
# The below function does division, with error handling for ZeroDivision.
def safe_divide(a, b):
    result = None
    try:
        result = a / b
    except ZeroDivisionError:
        result = None
    else:
        # Only runs if no exception occurred
        result = float(result)
    finally:
        # Always runs (e.g., log, cleanup)
        completed = True
    return result

In [ ]:
print(safe_divide(10, 2))  # 5.0
print(safe_divide(10, 0))  # None

---
### ✏️✏️ Exercise 3: Catching Specific Exceptions ✏️✏️
---

The below code asks the user for an index, then attempts to select that item from the list. Use except statements to handle the following errors:
- The user enters a non-integer input
- The user enters an index that is out of range of the list

In [ ]:
items = ["apple", "banana", "cherry", "date"]

idx_str = input("Enter an index (0-3): ")

try:
    # TODO: convert idx_str to int
    # TODO: get the item at that index
    pass
except: # TODO: Add exceptions for the two errors described above
    pass
else:
    # TODO: print the selected item
    pass
finally:
    # TODO: print a small 'done' message
    pass

## 2 - Audio in Python

Now we will look at loading, visualising and saving audio in Python. To start we need to install some additional packages, and import some packages.

In [ ]:
%pip install soundfile #This downloads and installs 'soundfile'
%pip install librosa   #This downloads and installs 'librosa'
import librosa
import soundfile as sf
import IPython
import numpy as np
import matplotlib.pyplot as plt

## 2.1 - Generating a Sine Wave

The below function generates a sine wave.

$y = Asin(2\pi ft + \phi)$

- The length argument determines the length of the sine wave in seconds
- The frequency argument determines the frequency of the sine wave in Hz
- The sample_rate argument determines the sample rate the sine wave is sampled at
- The phase argument determines the phase term of the sine wave, $\phi$
- The amplitude arguments determines the amplitude of the sine wave, $A$

In [ ]:
def sin_wave(length, frequency, sample_rate, phase = 0, amp = 1):
    n = int(round(length * sample_rate)) # Total number of samples
    t = np.arange(n) / float(sample_rate)  # 0, 1/sr, 2/sr ..., (n-1)/sr
    return amp*np.sin(2*np.pi*frequency*t + phase), t

Let's generate a sine wave, that is 2 seconds long, at a frequency of 100 Hz, and a sample rate of 44100.

In [ ]:
sine, tAx = sin_wave(2, 100, 44100)
print(f'the sine wave is a NumPy Array with shape: {sine.shape}')

In [ ]:
# Plot the first 1000 samples of the Sine wave, tAx is the time-axis
plt.plot(tAx[0:1000], sine[0:1000])
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')


IPython.display.Audio() allows you play an audio file right here in this notebook! call IPython.display.Audio() and provide the following arguments:

- 'data' - this should be a numpy array holding audio data
- 'rate' - this should be set to sample rate of the audio data


In [ ]:
IPython.display.Audio(data = sine, rate=44100)

This should allow you to listen to the sine wave. 

---
### ✏️✏️ Exercise 4: Generating and Summing Sine Waves ✏️✏️
---

Generate three sine waves, with the following parameters
1. $f_1 = 100$, $A_1 = 0.2$, $\phi_1 = \pi/4$
2. $f_2 = 200$, $A_2 = 0.3$, $\phi_2 = \pi/2$
3. $f_3 = 300$, $A_3 = 0.1$, $\phi_3 = 0$

Plot each sine wave and listen to each sine wave individually, then sum the sine waves together. Plot and listen to the resulting summed sinusoid.

For all sine waves, use the sample_rate of $fs=44100$.

In [ ]:
# Your code here

------
### 2.2 - Reading Audio Data from file

This worksheet should have come with an accompanying audio file repository, called 'Audio'. It contains some audio files. We can load an audio file using Librosa.

In [ ]:
bass_drum, sample_rate = librosa.load('Audio/Bass Sample 1.wav', mono=True, sr=None)
# mono=True converts to mono (one channel of audio)
# sr=None loads the file at whatever sample rate it was recorded at

In [ ]:
IPython.display.Audio(data = bass_drum, rate=sample_rate)
# Always be sure to set the right sample rate when playing back audio

An audio array is a time-series consisting of samples. The sample rate (measured in Hz) describes how many samples there are in one-second of audio. The total length of an audio array in seconds is:

length (in seconds) = number of samples/sample rate

So, the above file is:

In [ ]:
print(f'Audio/Bass Sample 1.wav is: {bass_drum.shape[0]/sample_rate} seconds long')

This information can be used to find the instant in time each sample of the waveform corresponds to:

In [ ]:
time = np.arange(0, bass_drum.shape[0])/sample_rate
print(time[0])
print(time[-1])
plt.plot(time, bass_drum)
plt.grid()
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')

---
### ✏️✏️ Exercise 5: Loading a Snare ✏️✏️
---

Load and play back one of the snare samples from the Audio directory. Plot the waveform, including the time-axis.

## 3 - Viewing the Spectrum

To view a spectrum in the frequency-domain, we can apply the *Fourier Transform*, or fft. In the frequency domain, the signal is also a vector of values, but now the values correspond to frequency instead of time. Here is an example:

In [ ]:
X = np.fft.rfft(bass_drum)  # Use np.fft.rfft() to carry out the fast fourier transform
print(X.shape)

### 3.1 - Taking the absolute values

The result of the fft is a vector of *complex numbers*. [Complex numbers](https://en.wikipedia.org/wiki/Complex_number) are beyond the scope of this course, but the key thing to understand is that each complex number represents a sinusoid, with *amplitude*, *frequency* and *phase*. We can discard the *phase* information and just look at the amplitude by taking the absolute value of the complex numbers:

In [ ]:
X_abs = np.abs(X)

In [ ]:
print(f'X is a complex numpy array, with dtype {X.dtype}')
print(f'X_abs is a numpy array, with dtype {X_abs.dtype}')

### 3.2 - Creating the frequency axis

Each element in the array X_abs represents the amplitude at a frequency. The precise value of these frequencies depends on the length of the time-domain signal the fft was applied to, and the sample rate.

NumPy has a function that creates this 'frequency-axis' for you, if you provide this information. These frequencies are sometimes referred to as fft *bin center frequencies*.

In [ ]:
signal_length = bass_drum.shape[0] # Length of time-domain signal in samples
sample_period = 1/sample_rate       # sample_period is given by 1/sample rate

f = np.fft.rfftfreq(bass_drum.shape[0], d=sample_period)

f is now a vector of evenly spaced values, representing frequency in Hz. The frequencies represented start at 0 and go all the way up until the *Nyquist frequency* - sample rate / 2

In [ ]:
f[0]

In [ ]:
f[-1] # Nyquist frequency, fs/2

### 3.3 - Plotting the result

Now we can plot our audio signal in the time and frequency domains!

In [ ]:
# Reload audio and recalculate spectrum
bass_drum, sample_rate = librosa.load('Audio/Bass Sample 1.wav', mono=True, sr=None)

N = bass_drum.shape[0] # Number of samples in audio file
time = np.arange(N) / samplerate # Time axis

X = np.fft.rfft(bass_drum)  
X_abs = np.abs(X)

sample_period = 1/sample_rate
f = np.fft.rfftfreq(bass_drum.shape[0], d=sample_period)

In [ ]:
plt.subplot(2, 1, 1) # Create a subplot
plt.grid()
plt.plot(time, bass_drum) # Plot in time-domain
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

plt.subplot(2, 1, 2) # Create another subplot
plt.grid()
plt.plot(f/1000, np.abs(X)) # Plot in frequency domain, converting to kHz
plt.xlabel("Frequency (KHz)")
plt.ylabel("Amplitude")

plt.show()

IPython.display.Audio(data = bass_drum, rate=sample_rate)

---
### ✏️✏️ Exercise 6: Snare Spectrum ✏️✏️
---

Load a snare sample. Plot it in the time-domain and in the frequency-domain.

## 4 - Feature Extraction

Now we will look at extracting audio features from our audio data.

### 4.1 - Zero-Crossing Rate

Below is a function that calculates the zero crossing rate from an audio vector:

In [ ]:
import numpy as np

def zero_crossing_rate(x):
    """
    Compute the zero-crossing rate for a 1D audio signal.
    
    Parameters
    ----------
    x : NumPy Array
        Audio signal. Shape (n_samples,).
        
    Returns
    -------
    float
        Zero-crossing rate (range [0, 1]).
    """
    x = np.asarray(x, dtype=float)

    x_sign = np.where(x>0, 1, 0)
    x_diff = np.diff(x_sign)
    return np.mean(np.abs(x_diff))

Run the below code to check it works! You can calculate the zero-crossing rate manually and compare it to the function output.

In [ ]:
random_x = np.random.randn(10)
zcr_1 = zero_crossing_rate(random_x)

In [ ]:
print(f'{random_x}\n has a zcr of: {zcr_1}')

### 4.2 - Spectral Centroid

Below is a function that calculates the Spectral Centroid from an audio vector:

In [ ]:
def spectral_centroid(x, sr):
    """
    Compute the spectral centroid of an audio signal, in Hz.
    The centroid is computed once for the entire signal.

    Formula:
        centroid = sum(f_k * |X_k|) / sum(|X_k|)
    where X_k is the FFT and f_k are bin center frequencies. |X| denotes the absolute of X

    Args:
        x : NumPy Array
            Audio signal. Shape (n_samples,).
        sr (int or float): Sampling rate in Hz.

    Returns:
        float: Spectral centroid in Hz.

    """
    # Real FFT and frequency bins
    X = np.fft.rfft(x)
    f = np.fft.rfftfreq(x.shape[0], d=1.0 / sr)

    # Take absolute
    X_abs = np.abs(X)

    # Find denominator
    denom = X_abs.sum()

    if denom <= 1e-12: # If input is silent, return 0
        return 0.0 
    
    # Calculate spectral centroid
    centroid_hz = (f * X_abs).sum() / denom
    return centroid_hz

In [ ]:
bass_drum, sample_rate = librosa.load('Audio/Bass Sample 1.wav', mono=True, sr=None)
spectral_centroid(bass_drum, sample_rate)

# 🎯🎯 Challenge! 🎯🎯

Now you have all the components required to load the audio files in the directory 'Audio', extract the features, and compare them by class.

---
Load the 10 bass drum samples, and the 10 snare drum samples, and extract audio features from them. The file names follow the pattern:
- 'Bass Sample n.wav'
- 'Snare Sample n.wav'

Use a for loop to load each audio file in turn, and:
- Load the audio file using librosa.load()
- Extract the features, ZCR and Spectral Centroid

This should result in:
 - 10 values of ZCR for the snare drums
 - 10 values of spectral centroid for the snare drums
 - 10 values of ZCR for the bass drums
 - 10 values of spectral centroid for the bass drums

Now, create a scatter plot of the two classes. The x-axis should show ZCR, and the y-axis should show Spectral Centroid. The two classes should be shown as different coloured markers.

---




In [ ]:
# Your code here!